# Make clean and cumulative field data for beech trees only (from all the data on the nextcloud, 2015 to 2023)

In [7]:
import pandas as pd
from pathlib import Path


# ============================================================
# 1. Dossier contenant les fichiers CSV
# ============================================================

dossier = Path("../data/Data_collecteurs_massane")  # <-- à adapter

fichiers = sorted(dossier.glob("Collecteurs_*.csv"))


# ============================================================
# 2. Colonnes de hêtre à récupérer
# ============================================================

colonnes_hetre = [
    "fag_feuil_ent",
    "fag_feuil_cas",
    "fag_feuil_gal",
    "fag_bourg",
    "fag_ecail",
    "fag_gal",
    "fag_enigm",
    "fag_ecorc",
    "fag_branch",
    "fag_fain_intac",
    "fag_fain_perc",
    "fag_bog",
    "fag_fleur",
    "fag_frui_avor",
    "fag_indet",
    "fag_necro"
]


# ============================================================
# 3. Correspondance mois -> numéro
# ============================================================

mois_dict = {
    "janvier": 1,
    "février": 2,
    "fevrier": 2,
    "mars": 3,
    "avril": 4,
    "mai": 5,
    "juin": 6,
    "juillet": 7,
    "août": 8,
    "aout": 8,
    "septembre": 9,
    "octobre": 10,
    "novembre": 11,
    "décembre": 12,
    "decembre": 12
}


# ============================================================
# 4. Parcourir tous les fichiers
# ============================================================

resultats = []

for fichier in fichiers:

    print(f"Lecture : {fichier.name}")

    # --------------------------------------------------------
    # Lire le CSV
    # --------------------------------------------------------

    df = pd.read_csv(
        fichier,
        sep=";",
        decimal=",",
        keep_default_na=True
    )

    # --------------------------------------------------------
    # Garder uniquement les lignes "_nb"
    # --------------------------------------------------------

    df = df[
        df["Collecteur"].str.endswith("_nb")
    ].copy()

    # --------------------------------------------------------
    # Récupérer l'identité de la station
    #
    # VH-C01_nb -> VH-C01
    # JH-C03_nb -> JH-C03
    # --------------------------------------------------------

    df["station"] = (
        df["Collecteur"]
        .str.replace("_nb", "", regex=False)
    )

    # --------------------------------------------------------
    # Garder uniquement les stations JH et VH
    # --------------------------------------------------------

    df = df[
        df["station"].str.startswith(("JH-", "VH-"))
    ].copy()

    # --------------------------------------------------------
    # Vérifier que toutes les colonnes existent
    # --------------------------------------------------------

    for col in colonnes_hetre:

        if col not in df.columns:

            print(
                f"  Attention : {col} absent de {fichier.name}"
            )

            df[col] = pd.NA

    # --------------------------------------------------------
    # Conversion en numérique
    #
    # Cellule vide -> NA
    # Valeur 0 -> 0
    # --------------------------------------------------------

    for col in colonnes_hetre:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Extraire mois et année du nom du fichier
    #
    # Exemple :
    # Collecteurs_avril_2025.csv
    # --------------------------------------------------------

    morceaux = fichier.stem.split("_")

    mois_nom = morceaux[-2].lower()
    annee = int(morceaux[-1])

    if mois_nom not in mois_dict:

        raise ValueError(
            f"Mois inconnu dans le fichier : {fichier.name}"
        )

    mois = mois_dict[mois_nom]

    # --------------------------------------------------------
    # Créer une seule colonne date
    #
    # avril 2025 -> 2025-04-01
    # --------------------------------------------------------

    df["date"] = pd.to_datetime(
        f"{annee}-{mois:02d}-01"
    )

    # --------------------------------------------------------
    # Garder uniquement les colonnes voulues
    # --------------------------------------------------------

    colonnes_finales = [
        "date",
        "station"
    ] + colonnes_hetre

    resultats.append(
        df[colonnes_finales]
    )


# ============================================================
# 5. Fusionner tous les fichiers
# ============================================================

if not resultats:

    raise ValueError(
        "Aucun fichier Collecteurs_*.csv trouvé."
    )

resultat_final = pd.concat(
    resultats,
    ignore_index=True
)


# ============================================================
# 6. Trier par date puis par station
# ============================================================

resultat_final = resultat_final.sort_values(
    ["date", "station"]
).reset_index(drop=True)


# ============================================================
# 7. Afficher
# ============================================================

print(resultat_final)


# ============================================================
# 8. Sauvegarder
# ============================================================

# sortie = dossier / "collecteurs_massane_JH_VH_2015_2023.csv"
#
# resultat_final.to_csv(
#     sortie,
#     sep=";",
#     index=False,
#     date_format="%Y-%m-%d"
# )
#
# print(f"\nFichier sauvegardé : {sortie}")

Lecture : Collecteurs_avril_2017.csv
Lecture : Collecteurs_avril_2018.csv
Lecture : Collecteurs_avril_2019.csv
Lecture : Collecteurs_avril_2020.csv
Lecture : Collecteurs_avril_2021.csv
Lecture : Collecteurs_avril_2022.csv
Lecture : Collecteurs_avril_2023.csv
Lecture : Collecteurs_decembre_2016.csv
Lecture : Collecteurs_decembre_2017.csv
Lecture : Collecteurs_decembre_2018.csv
Lecture : Collecteurs_decembre_2019.csv
Lecture : Collecteurs_decembre_2020.csv
Lecture : Collecteurs_decembre_2021.csv
Lecture : Collecteurs_decembre_2022.csv
Lecture : Collecteurs_decembre_2023.csv
Lecture : Collecteurs_juillet_2017.csv
Lecture : Collecteurs_juillet_2018.csv
Lecture : Collecteurs_juillet_2019.csv
Lecture : Collecteurs_juillet_2020.csv
Lecture : Collecteurs_juillet_2021.csv
Lecture : Collecteurs_juillet_2022.csv
Lecture : Collecteurs_juillet_2023.csv
Lecture : Collecteurs_juin_2016.csv
Lecture : Collecteurs_juin_2017.csv
Lecture : Collecteurs_juin_2018.csv
Lecture : Collecteurs_juin_2019.csv
Lect